# Kaggle Utilities — Project Ouroboros

Thin Kaggle orchestration notebook for the packaged Ouroboros entrypoints. It can:

- hydrate the persistent Hugging Face cache from the attached `ouroboros-cache` dataset
- load Kaggle secrets safely into environment variables
- pull the latest repo files from GitHub into `/kaggle/working`
- launch training with Kaggle/IPython `!torchrun` shell magic so launch failures surface promptly instead of being hidden behind a long-running Python subprocess

Reusable training, checkpoint, DGAC, DiLoCo worker, and command-contract behavior lives in `ouroboros/*`; this notebook should remain a thin adapter.


In [1]:
import os
import shutil
from pathlib import Path

!cp -r /kaggle/input/datasets/weirdrunner007/ouroboros-cache/* /kaggle/working/

os.environ["HF_HOME"] = "/kaggle/working/hf_cache"

In [2]:
from __future__ import annotations

import os

try:
    from kaggle_secrets import UserSecretsClient
except Exception:
    UserSecretsClient = None


def _normalize_text(value: object | None, *, uppercase: bool = False) -> str | None:
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    return text.upper() if uppercase else text


_secret_client = None


def _get_secret_client():
    global _secret_client
    if _secret_client is False:
        return None
    if _secret_client is None:
        if UserSecretsClient is None:
            _secret_client = False
            return None
        try:
            _secret_client = UserSecretsClient()
        except Exception:
            _secret_client = False
            return None
    return _secret_client if _secret_client is not False else None


def _optional_secret(name: str) -> str | None:
    client = _get_secret_client()
    if client is None:
        return None
    try:
        value = client.get_secret(name)
    except Exception:
        return None
    return _normalize_text(value)


def _resolve_value(
    env_names: tuple[str, ...],
    secret_names: tuple[str, ...] = (),
    *,
    uppercase: bool = False,
) -> str | None:
    for env_name in env_names:
        value = _normalize_text(os.environ.get(env_name), uppercase=uppercase)
        if value is not None:
            return value
    for secret_name in secret_names:
        value = _normalize_text(_optional_secret(secret_name), uppercase=uppercase)
        if value is not None:
            return value
    return None


HF_TOKEN = _resolve_value(("HF_TOKEN", "HUGGINGFACE_HUB_TOKEN"), ("HF_TOKEN",))
WANDB_KEY = _resolve_value(("WANDB_API_KEY", "WANDB_KEY"), ("WANDB_KEY",))
GITHUB_TOKEN = _resolve_value(("GITHUB_TOKEN", "GH_TOKEN"), ("GITHUB_TOKEN", "GH_TOKEN"))
DILOCO_WORKER_ID = _resolve_value(
    ("DILOCO_WORKER_ID", "OUROBOROS_DILOCO_WORKER_ID", "WORKER_ID"),
    ("DILOCO_WORKER_ID",),
    uppercase=True,
)

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
if WANDB_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_KEY
    os.environ["WANDB_KEY"] = WANDB_KEY
if GITHUB_TOKEN:
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
    os.environ["GH_TOKEN"] = GITHUB_TOKEN
if DILOCO_WORKER_ID:
    os.environ["DILOCO_WORKER_ID"] = DILOCO_WORKER_ID
    os.environ.setdefault("OUROBOROS_DILOCO_WORKER_ID", DILOCO_WORKER_ID)
    os.environ.setdefault("WORKER_ID", DILOCO_WORKER_ID)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print({
    "HF_TOKEN": bool(os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")),
    "WANDB_KEY": bool(os.environ.get("WANDB_API_KEY") or os.environ.get("WANDB_KEY")),
    "GITHUB_TOKEN": bool(os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")),
    "DILOCO_WORKER_ID": bool(
        os.environ.get("DILOCO_WORKER_ID")
        or os.environ.get("OUROBOROS_DILOCO_WORKER_ID")
        or os.environ.get("WORKER_ID")
    ),
})


In [3]:
from __future__ import annotations

import base64
import json
import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

DEFAULT_REPO_URL = "https://github.com/deveshpat/Ouroboros.git"
DEFAULT_REPO_REF = "main"

REPO_URL = (os.environ.get("OUROBOROS_REPO_URL") or DEFAULT_REPO_URL).strip()
REPO_REF = (os.environ.get("OUROBOROS_REPO_REF") or DEFAULT_REPO_REF).strip()
REPO_COMMIT = (os.environ.get("OUROBOROS_REPO_COMMIT") or "").strip()
REPO_DIR = Path("/kaggle/working/ouroboros_repo")
TARGET_DIR = Path("/kaggle/working")
FILES_TO_COPY = [
    "-m ouroboros.coconut",
    "ouroboros/",
]


def run(
    cmd: list[str],
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
    *,
    check: bool = True,
) -> subprocess.CompletedProcess:
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=check,
        text=True,
        capture_output=True,
    )


def git_env(repo_url: str) -> dict[str, str]:
    env = os.environ.copy()
    token = (os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN") or "").strip()
    if token and repo_url.startswith("https://") and "github.com" in repo_url:
        basic = base64.b64encode(f"x-access-token:{token}".encode("utf-8")).decode("ascii")
        env["GIT_CONFIG_COUNT"] = "1"
        env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
        env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {basic}"
    return env


def ensure_repo(repo_url: str, repo_dir: Path, env: dict[str, str]) -> None:
    if not repo_dir.exists():
        run(["git", "clone", "--filter=blob:none", repo_url, str(repo_dir)], env=env)
    else:
        run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, env=env)
        run(["git", "clean", "-fd"], cwd=repo_dir, env=env)


def fetch_and_checkout(repo_dir: Path, ref: str, commit: str, env: dict[str, str]) -> str:
    ref = (ref or "").strip()
    commit = (commit or "").strip()

    fetched = False
    if ref:
        for fetch_cmd in (
            ["git", "fetch", "--depth", "1", "origin", ref],
            ["git", "fetch", "--depth", "1", "origin", f"refs/heads/{ref}"],
            ["git", "fetch", "--depth", "1", "origin", f"refs/tags/{ref}"],
        ):
            result = run(fetch_cmd, cwd=repo_dir, env=env, check=False)
            if result.returncode == 0:
                fetched = True
                break

    if not fetched:
        run(["git", "fetch", "origin"], cwd=repo_dir, env=env)

    run(["git", "checkout", "--force", "--detach", "FETCH_HEAD"], cwd=repo_dir, env=env)

    if commit:
        commit_result = run(["git", "checkout", "--force", "--detach", commit], cwd=repo_dir, env=env, check=False)
        if commit_result.returncode != 0:
            run(["git", "fetch", "origin"], cwd=repo_dir, env=env)
            commit_result = run(["git", "checkout", "--force", "--detach", commit], cwd=repo_dir, env=env, check=False)
            if commit_result.returncode != 0:
                stderr = (commit_result.stderr or commit_result.stdout or "").strip()
                raise RuntimeError(f"Failed to checkout OUROBOROS_REPO_COMMIT={commit!r}: {stderr}")

    return run(["git", "rev-parse", "HEAD"], cwd=repo_dir, env=env).stdout.strip()


def copy_repo_files(repo_dir: Path, target_dir: Path, file_names: list[str]) -> list[str]:
    copied: list[str] = []
    for name in file_names:
        src = repo_dir / name.rstrip("/")
        if not src.exists():
            print(f"[warn] missing in repo: {name}")
            continue
        dst = target_dir / src.name
        if src.is_dir():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        copied.append(src.name + ("/" if src.is_dir() else ""))
    return copied


git_environment = git_env(REPO_URL)
ensure_repo(REPO_URL, REPO_DIR, git_environment)
commit = fetch_and_checkout(REPO_DIR, REPO_REF, REPO_COMMIT, git_environment)
copied = copy_repo_files(REPO_DIR, TARGET_DIR, FILES_TO_COPY)
metadata = {
    "repo_ref": REPO_REF,
    "requested_commit": REPO_COMMIT or None,
    "commit": commit,
    "copied_files": copied,
    "synced_at_utc": datetime.now(timezone.utc).isoformat(),
}
(Path("/kaggle/working/repo_sync_metadata.json")).write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))


In [ ]:
from __future__ import annotations

import os

from ouroboros.coordinator.kaggle_commands import (
    format_shell_command,
    kaggle_secret_presence,
    resolve_diloco_worker_id,
    resolve_kaggle_run_mode,
)
from ouroboros.coordinator.kaggle_launch_matrix import (
    apply_launch_environment_defaults,
    build_launch_command,
    get_launch_spec,
)

print(kaggle_secret_presence(os.environ))
run_mode = resolve_kaggle_run_mode(os.environ)
os.environ["OUROBOROS_KAGGLE_RUN_MODE"] = run_mode
launch_spec = get_launch_spec(run_mode)
print({
    "OUROBOROS_KAGGLE_RUN_MODE": run_mode,
    "workflow_label": launch_spec.workflow_label,
    "output_env_key": launch_spec.output_env_key,
})

apply_launch_environment_defaults(run_mode, os.environ)

worker_id = None
if launch_spec.requires_worker_id:
    worker_id = resolve_diloco_worker_id(os.environ)
    os.environ["DILOCO_WORKER_ID"] = worker_id
    print({"DILOCO_WORKER_ID": worker_id})

command = build_launch_command(run_mode, os.environ, worker_id=worker_id)
shell_command = format_shell_command(command)
print("[launch] " + shell_command)

# Keep the actual launch as Kaggle/IPython shell magic, not subprocess.run.
# The command argv comes from ouroboros.coordinator.kaggle_launch_matrix; do not duplicate it here.
!{shell_command}
